In [15]:
import json
import os
import time
import soccerdata as sd

import pandas as pd
import requests
from dotenv import load_dotenv

[04/28/26 14:31:15] INFO     No custom team name replacements found. You can configure these in       ]8;id=12510991;file://c:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-packages\soccerdata\_config.py\_config.py]8;;\:]8;id=12510992;file://c:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-packages\soccerdata\_config.py#92\92]8;;\
                             C:\Users\MUEL\soccerdata\config\teamname_replacements.json.                           

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=12510998;file://c:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-packages\soccerdata\_config.py\_config.py]8;;\:]8;id=12510999;file://c:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-packages\soccerdata\_config.py#190\190]8;;\
                             C:\Users\MUEL\soccerdata\config\league_dict.json.                                     

In [3]:
load_dotenv()

BASE_URL = "https://sports.bzzoiro.com/api"
TOKEN = os.environ["BZZOIRO_TOKEN"]
HEADERS = {"Authorization": f"Token {TOKEN}"}

PL_LEAGUE_ID = 1
PL_SEASONS = [336, 337]  # 24/25, 25/26

os.makedirs("data", exist_ok=True)

In [12]:
def fetch_all(endpoint, params=None, verbose=True):
    params = dict(params or {})
    params["limit"] = 200
    results, offset = [], 0
    while True:
        params["offset"] = offset
        r = requests.get(f"{BASE_URL}/{endpoint}/", headers=HEADERS, params=params, timeout=30)
        print(r.url)
        r.raise_for_status()
        data = r.json()
        batch = data.get("results", [])
        results.extend(batch)
        if verbose:
            print(f"  {endpoint}: {len(results)}/{data.get('count', '?')}")
        if not data.get("next"):
            break
        offset += 200
        time.sleep(0.2)
    return results

In [5]:
def fetch_all_pages(endpoint, params=None, verbose=True):
    params = dict(params or {})
    params["page_size"] = 500
    results, page = [], 1
    while True:
        params["page"] = page
        r = requests.get(f"{BASE_URL}/{endpoint}/", headers=HEADERS, params=params, timeout=30)
        r.raise_for_status()
        data = r.json()
        batch = data.get("results", [])
        results.extend(batch)
        if verbose:
            print(f"  {endpoint}: {len(results)}/{data.get('count', '?')}")
        if not data.get("next"):
            break
        page += 1
        time.sleep(0.2)
    return results

## PL_events

In [10]:
def flatten_event(ev):
    row = dict(ev)
    for key, fields in [
        ("league",       ["id", "name", "slug"]),
        ("season",       ["id", "name", "year"]),
        ("home_team_obj",["id", "name", "short_name"]),
        ("away_team_obj",["id", "name", "short_name"]),
        ("home_coach",   ["id", "name"]),
        ("away_coach",   ["id", "name"]),
        ("referee",      ["id", "name"]),
        ("venue",        ["id", "name", "city", "capacity"]),
    ]:
        obj = row.pop(key, None) or {}
        for f in fields:
            row[f"{key}_{f}"] = obj.get(f)

    for key in ("unavailable_players", "lineups", "shotmap", "momentum"):
        val = row.get(key)
        row[key] = json.dumps(val, ensure_ascii=False) if val is not None else None

    odds = row.pop("odds", None) or {}
    row["odds_home"] = odds.get("home")
    row["odds_draw"] = odds.get("draw")
    row["odds_away"] = odds.get("away")
    return row

In [13]:
all_events = []
for season_id in PL_SEASONS:
    print(f"Season {season_id}...")
    all_events += fetch_all("events", {"season": season_id, "full": "true"})

pl_events = pd.DataFrame([flatten_event(ev) for ev in all_events])
pl_events.to_csv("data/pl_events.csv", index=False)
print(f"\npl_events: {pl_events.shape}")
pl_events.head(2)

Season 336...
https://sports.bzzoiro.com/api/events/?season=336&full=true&limit=200&offset=0
  events: 200/380
https://sports.bzzoiro.com/api/events/?season=336&full=true&limit=200&offset=200
  events: 380/380
Season 337...
https://sports.bzzoiro.com/api/events/?season=337&full=true&limit=200&offset=0
  events: 200/385
https://sports.bzzoiro.com/api/events/?season=337&full=true&limit=200&offset=200
  events: 385/385

pl_events: (765, 60)


,id,home_team,away_team,event_date,round_number,group_name,status,home_score,away_score,home_score_ht,...,home_coach_id,home_coach_name,away_coach_id,away_coach_name,referee_id,referee_name,venue_id,venue_name,venue_city,venue_capacity
0,160173,Manchester United,Fulham,2024-08-16T20:00:00+04:00,1,None,finished,1.0,0.0,NaN,...,NaN,NaN,NaN,NaN,None,NaN,17,Old Trafford,Manchester,74879
1,160174,Ipswich Town,Liverpool,2024-08-17T12:30:00+04:00,1,None,finished,0.0,2.0,NaN,...,NaN,NaN,NaN,NaN,None,NaN,195,Portman Road Stadium,Ipswich,29673


## Managers

In [8]:
def flatten_manager(m):
    row = dict(m)
    team = row.pop("current_team", None) or {}
    row["current_team_id"] = team.get("id")
    row["current_team_name"] = team.get("name")
    row["tactical_styles"] = json.dumps(row.get("tactical_styles", []), ensure_ascii=False)
    return row

In [9]:
managers_raw = fetch_all_pages("managers")
managers = pd.DataFrame([flatten_manager(m) for m in managers_raw])
managers.to_csv("data/managers.csv", index=False)
print(f"managers: {managers.shape}")
managers.head(2)

  managers: 50/1373
  managers: 100/1373
  managers: 150/1373
  managers: 200/1373
  managers: 250/1373
  managers: 300/1373
  managers: 350/1373
  managers: 400/1373
  managers: 450/1373
  managers: 500/1373
  managers: 550/1373
  managers: 600/1373


KeyboardInterrupt: 

## Referees

In [6]:
referees_raw = fetch_all("referees")
referees = pd.DataFrame(referees_raw)
referees.to_csv("data/referees.csv", index=False)
print(f"referees: {referees.shape}")
referees.head(2)

  referees: 971/971
referees: (971, 12)


,name,country,matches,avg_yellow_per_match,avg_red_per_match,avg_goals_per_match,avg_fouls_per_match,total_yellow_cards,total_red_cards,career_games,career_yellow_cards,career_red_cards
0,Damaso Arcediano Monescillo,Spain,45,5.29,0.13,2.47,28.7,238,6,None,1797.0,40.0
1,Jon Ander Gonzalez Esteban,Spain,44,4.45,0.25,2.70,24.6,196,11,None,707.0,39.0


## FBref

### 2024

In [17]:
fbref = sd.FBref('ENG-Premier League', '2024')

[04/28/26 14:34:13] INFO     Saving cached data to C:\Users\MUEL\soccerdata\data\FBref               ]8;id=12511012;file://c:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=12511013;file://c:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-packages\soccerdata\_common.py#250\250]8;;\

[04/28/26 14:34:43] ERROR    Error while scraping                                                    ]8;id=12511019;file://c:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=12511020;file://c:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-packages\soccerdata\_common.py#623\623]8;;\
                             https://fbref.com/en/comps/9/2024-2025/2024-2025-Premier-League-Stats.                
                             Retrying in 0 seconds... (attempt 1 of 5).                                            
                             Traceback (most recent call last):                                                    
                               File                                                                                
                             "c:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-packa               
                             ges\soccerdata\_common.py", line 607, in _download_and_save                           
                                 response = self._validate_page(url).encode("utf-8")                               
                                            ~~~~~~~~~~~~~~~~~~~^^^^^                                               
                               File                                                                                
                             "c:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-packa               
                             ges\soccerdata\fbref.py", line 1031, in _validate_page                                
                                 raise Exception(                                                                  
                                 ...<2 lines>...                                                                   
                                 )                                                                                 
                             Exception: Could not retrieve page content within timeout. Possible                   
                             reasons: failed CAPTCHA, IP block or network issues.                                  

In [ ]:
games = fbref.read_schedule()
games.to_csv("data/fbref_games2024.csv", index=False)

In [19]:
lineup = fbref.read_lineup()
lineup.to_csv("data/fbref_lineup2024.csv", index=False)

[04/28/26 14:36:57] INFO     [1/380] Retrieving game with id=cc5b4244                                  ]8;id=12511027;file://c:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-packages\soccerdata\fbref.py\fbref.py]8;;\:]8;id=12511028;file://c:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-packages\soccerdata\fbref.py#854\854]8;;\

[04/28/26 14:37:09] WARNING  c:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-packa ]8;id=12511035;file://C:\Users\MUEL\AppData\Local\Programs\Python\Python313\Lib\warnings.py\warnings.py]8;;\:]8;id=12511036;file://C:\Users\MUEL\AppData\Local\Programs\Python\Python313\Lib\warnings.py#110\110]8;;\
                             ges\soccerdata\fbref.py:880: FutureWarning: This search incorrectly                   
                             ignores the root element, and will be fixed in a future version.  If                  
                             you rely on the current behaviour, change it to ".//table"                            
                               html_stats_table = tree.find(                                                       
                                                                                                                   

                    WARNING  c:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-packa ]8;id=12511041;file://C:\Users\MUEL\AppData\Local\Programs\Python\Python313\Lib\warnings.py\warnings.py]8;;\:]8;id=12511042;file://C:\Users\MUEL\AppData\Local\Programs\Python\Python313\Lib\warnings.py#110\110]8;;\
                             ges\soccerdata\fbref.py:880: FutureWarning: This search incorrectly                   
                             ignores the root element, and will be fixed in a future version.  If                  
                             you rely on the current behaviour, change it to ".//table"                            
                               html_stats_table = tree.find(                                                       
                                                                                                                   

                    INFO     [2/380] Retrieving game with id=c0e3342a                                  ]8;id=12511047;file://c:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-packages\soccerdata\fbref.py\fbref.py]8;;\:]8;id=12511048;file://c:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-packages\soccerdata\fbref.py#854\854]8;;\

[04/28/26 14:37:22] WARNING  c:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-packa ]8;id=12511053;file://C:\Users\MUEL\AppData\Local\Programs\Python\Python313\Lib\warnings.py\warnings.py]8;;\:]8;id=12511054;file://C:\Users\MUEL\AppData\Local\Programs\Python\Python313\Lib\warnings.py#110\110]8;;\
                             ges\soccerdata\fbref.py:880: FutureWarning: This search incorrectly                   
                             ignores the root element, and will be fixed in a future version.  If                  
                             you rely on the current behaviour, change it to ".//table"                            
                               html_stats_table = tree.find(                                                       
                                                                                                                   

...

[04/28/26 15:56:20] WARNING  c:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-packa ]8;id=12517839;file://C:\Users\MUEL\AppData\Local\Programs\Python\Python313\Lib\warnings.py\warnings.py]8;;\:]8;id=12517840;file://C:\Users\MUEL\AppData\Local\Programs\Python\Python313\Lib\warnings.py#110\110]8;;\
                             ges\soccerdata\fbref.py:880: FutureWarning: This search incorrectly                   
                             ignores the root element, and will be fixed in a future version.  If                  
                             you rely on the current behaviour, change it to ".//table"                            
                               html_stats_table = tree.find(                                                       
                                                                                                                   

                    WARNING  c:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-packa ]8;id=12517845;file://C:\Users\MUEL\AppData\Local\Programs\Python\Python313\Lib\warnings.py\warnings.py]8;;\:]8;id=12517846;file://C:\Users\MUEL\AppData\Local\Programs\Python\Python313\Lib\warnings.py#110\110]8;;\
                             ges\soccerdata\fbref.py:880: FutureWarning: This search incorrectly                   
                             ignores the root element, and will be fixed in a future version.  If                  
                             you rely on the current behaviour, change it to ".//table"                            
                               html_stats_table = tree.find(                                                       
                                                                                                                   

                    INFO     [380/380] Retrieving game with id=812ef8ad                                ]8;id=12517851;file://c:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-packages\soccerdata\fbref.py\fbref.py]8;;\:]8;id=12517852;file://c:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-packages\soccerdata\fbref.py#854\854]8;;\

[04/28/26 15:56:32] WARNING  c:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-packa ]8;id=12517857;file://C:\Users\MUEL\AppData\Local\Programs\Python\Python313\Lib\warnings.py\warnings.py]8;;\:]8;id=12517858;file://C:\Users\MUEL\AppData\Local\Programs\Python\Python313\Lib\warnings.py#110\110]8;;\
                             ges\soccerdata\fbref.py:880: FutureWarning: This search incorrectly                   
                             ignores the root element, and will be fixed in a future version.  If                  
                             you rely on the current behaviour, change it to ".//table"                            
                               html_stats_table = tree.find(                                                       
                                                                                                                   

                    WARNING  c:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-packa ]8;id=12517863;file://C:\Users\MUEL\AppData\Local\Programs\Python\Python313\Lib\warnings.py\warnings.py]8;;\:]8;id=12517864;file://C:\Users\MUEL\AppData\Local\Programs\Python\Python313\Lib\warnings.py#110\110]8;;\
                             ges\soccerdata\fbref.py:880: FutureWarning: This search incorrectly                   
                             ignores the root element, and will be fixed in a future version.  If                  
                             you rely on the current behaviour, change it to ".//table"                            
                               html_stats_table = tree.find(                                                       
                                                                                                                   

In [20]:
team_season_stats = fbref.read_team_season_stats()
team_season_stats.to_csv("data/fbref_team_season_stats2024.csv", index=False)

In [21]:
player_season_stats = fbref.read_player_season_stats()
player_season_stats.to_csv("data/fbref_player_season_stats2024.csv", index=False)

### 2025

In [22]:
fbref = sd.FBref('ENG-Premier League', '2025')

[04/28/26 16:23:46] INFO     Saving cached data to C:\Users\MUEL\soccerdata\data\FBref               ]8;id=12517869;file://c:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=12517870;file://c:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-packages\soccerdata\_common.py#250\250]8;;\

In [24]:
games = fbref.read_schedule()
games.to_csv("data/fbref_games2025.csv", index=False)

In [ ]:
lineup = fbref.read_lineup()
lineup.to_csv("data/fbref_lineup2025.csv", index=False)

In [25]:
team_season_stats = fbref.read_team_season_stats()
team_season_stats.to_csv("data/fbref_team_season_stats2025.csv", index=False)

In [ ]:
player_season_stats = fbref.read_player_season_stats()
player_season_stats.to_csv("data/fbref_player_season_stats2025.csv", index=False)

## Elo

In [26]:
clubelo = sd.ClubElo("ENG-Premier League", "2024")

[04/28/26 16:28:04] INFO     Saving cached data to C:\Users\MUEL\soccerdata\data\ClubElo             ]8;id=12517875;file://c:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=12517876;file://c:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-packages\soccerdata\_common.py#250\250]8;;\

[2026-04-28 16:28:04] INFO     TLSLibrary:load:435 - Downloading required library version v1.13.1...


                    INFO     Downloading required library version v1.13.1...                       ]8;id=12517883;file://c:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-packages\tls_requests\models\libraries.py\libraries.py]8;;\:]8;id=12517884;file://c:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-packages\tls_requests\models\libraries.py#435\435]8;;\

[2026-04-28 16:28:04] INFO     TLSLibrary:download:314 - System Info - Platform: windows, Machine: amd64, File Ext : dll.


                    INFO     System Info - Platform: windows, Machine: amd64, File Ext : dll.      ]8;id=12517890;file://c:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-packages\tls_requests\models\libraries.py\libraries.py]8;;\:]8;id=12517891;file://c:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-packages\tls_requests\models\libraries.py#314\314]8;;\

...

                    INFO     Fetched release data from GitHub API.                                 ]8;id=12517904;file://c:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-packages\tls_requests\models\libraries.py\libraries.py]8;;\:]8;id=12517905;file://c:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-packages\tls_requests\models\libraries.py#239\239]8;;\

[2026-04-28 16:28:05] INFO     TLSLibrary:download:328 - Trying to download library from: https://github.com/bogdanfinn/tls-client/releases/download/v1.13.1/tls-client-xgo-1.13.1-windows-amd64.dll


                    INFO     Trying to download library from:                                      ]8;id=12517911;file://c:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-packages\tls_requests\models\libraries.py\libraries.py]8;;\:]8;id=12517912;file://c:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-packages\tls_requests\models\libraries.py#328\328]8;;\
                             https://github.com/bogdanfinn/tls-client/releases/download/v1.13.1/tl                 
                             s-client-xgo-1.13.1-windows-amd64.dll                                                 

[2026-04-28 16:28:08] INFO     TLSLibrary:_load_library:397 - Successfully loaded TLS library: C:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-packages\tls_requests\bin\tls-client-xgo-1.13.1-windows-amd64.dll


[04/28/26 16:28:08] INFO     Successfully loaded TLS library:                                      ]8;id=12517918;file://c:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-packages\tls_requests\models\libraries.py\libraries.py]8;;\:]8;id=12517919;file://c:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-packages\tls_requests\models\libraries.py#397\397]8;;\
                             C:\Users\MUEL\Desktop\INSA\IF\4A\INSA\SMART\QSport\venv\Lib\site-pack                 
                             ages\tls_requests\bin\tls-client-xgo-1.13.1-windows-amd64.dll                         

In [45]:
import io
clubs_anglais = [
    "Arsenal", 
    "Man City", 
    "Liverpool", 
    "Aston Villa", 
    "Man United", 
    "Brighton", 
    "Bournemouth", 
    "Chelsea", 
    "Brentford", 
    "Newcastle", 
    "Forest", 
    "Everton", 
    "Fulham", 
    "Crystal Palace", 
    "Leeds", 
    "West Ham", 
    "Tottenham", 
    "Sunderland", 
    "Wolves", 
    "Burnley", 
    "Coventry", 
    "Ipswich", 
    "Southampton", 
    "Millwall", 
    "Middlesbrough", 
    "Norwich", 
    "Derby", 
    "Sheffield United", 
    "Swansea", 
    "Wrexham", 
    "Hull", 
    "West Brom", 
    "Birmingham", 
    "Blackburn", 
    "Portsmouth", 
    "Bristol City", 
    "Leicester", 
    "QPR", 
    "Preston", 
    "Oxford", 
    "Watford", 
    "Stoke", 
    "Charlton", 
    "Sheffield Weds"
]
results = pd.DataFrame()
for club in clubs_anglais:
    club_formate = club.replace(" ", "")
    response = requests.get(f"http://api.clubelo.com/{club_formate}")
    if response.status_code == 200:
        csv_en_memoire = io.StringIO(response.text)
        df_club = pd.read_csv(csv_en_memoire)
        df_club = df_club[df_club["From"] >= "2024-08-16"]
        print(df_club.head(2))
        results = pd.concat([results, df_club], ignore_index=True)
        print(results.head(2))
    else :
        print(f"Erreur pour le club {club} : {response.status_code}")
results.to_csv("data/clubelo.csv", index=False)
    

      Rank     Club Country  Level          Elo        From          To
6324   4.0  Arsenal     ENG      1  1949.485474  2024-08-18  2024-08-22
6325   4.0  Arsenal     ENG      1  1949.749512  2024-08-23  2024-08-24
      Rank      Club Country  Level          Elo        From          To
6138   1.0  Man City     ENG      1  2055.707031  2024-08-19  2024-08-22
6139   1.0  Man City     ENG      1  2055.970947  2024-08-23  2024-08-24
      Rank       Club Country  Level          Elo        From          To
5773   6.0  Liverpool     ENG      1  1903.844971  2024-08-18  2024-08-22
5774   6.0  Liverpool     ENG      1  1904.109131  2024-08-23  2024-08-25
      Rank         Club Country  Level          Elo        From          To
6791  29.0  Aston Villa     ENG      1  1778.662476  2024-08-18  2024-08-18
6792  28.0  Aston Villa     ENG      1  1778.662476  2024-08-19  2024-08-20
      Rank        Club Country  Level          Elo        From          To
6015  27.0  Man United     ENG      1  1

## Météo

In [63]:
import pandas as pd
import requests
import time
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# ==========================================
# 1. INITIALISATION DU GÉOCODEUR & CACHE
# ==========================================
geolocator = Nominatim(user_agent="football_weather_pipeline_fr")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.1) # Respect strict de l'API

# Ce dictionnaire va stocker les coordonnées pour ne chercher chaque club qu'une seule fois
cache_coordonnees = {}

def obtenir_coordonnees(club, ville):
    """Récupère les coordonnées GPS d'un club, avec système de cache."""
    # Si on a déjà cherché ce club, on renvoie directement le résultat (gain de temps énorme)
    if club in cache_coordonnees:
        return cache_coordonnees[club]
    
    try:
        # Essai 1 : Nom du club + Stadium + UK
        location = geolocator.geocode(f"{club} Stadium, United Kingdom")
        
        # Essai 2 (Fallback) : Juste le nom du club/ville + UK
        if location is None:
            location = geolocator.geocode(f"{ville}, United Kingdom")
            
        if location:
            print(f"[GPS] Trouvé : {club} -> {location.latitude}, {location.longitude}", end="\r")
            # On sauvegarde dans le cache
            cache_coordonnees[club] = (location.latitude, location.longitude)
            time.sleep(1) # Pause pour OpenStreetMap
            return location.latitude, location.longitude
        else:
            print(f"[GPS] Introuvable pour : {club}")
            cache_coordonnees[club] = (None, None)
            return None, None
            
    except Exception as e:
        print(f"[GPS] Erreur API pour {club} : {e}")
        return None, None

# ==========================================
# 2. FONCTION MÉTÉO (OPEN-METEO)
# ==========================================
def obtenir_meteo_match_complete(club_domicile, club_exterieur, date_m, lat, lon, date_match, time_match):
    """Récupère toutes les variables météo utiles pour un match de foot."""
    if pd.isna(lat) or pd.isna(lon):
        return {} # Retourne un dictionnaire vide si pas de GPS
        
    url = "https://archive-api.open-meteo.com/v1/archive"
        
    # Liste enrichie des variables
    variables_meteo = "temperature_2m,relative_humidity_2m,precipitation,snowfall,wind_speed_10m,wind_gusts_10m"
        
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": date_match,
        "end_date": date_match,
        "hourly": variables_meteo,
        "timezone": "Europe/London"
    }
    
    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        data = response.json()
        df_meteo = pd.DataFrame(data['hourly'])
        df_meteo['time'] = pd.to_datetime(df_meteo['time'])
        
        kickoff = pd.to_datetime(f"{date_match} {time_match}")
        full_time = kickoff + pd.Timedelta(hours=2)
        
        df_match = df_meteo[(df_meteo['time'] >= kickoff) & (df_meteo['time'] <= full_time)]
        
        # Si pour une raison quelconque on n'a pas de données sur cette tranche horaire
        if df_match.empty:
            return {}
            
        # Agrégration mathématique logique selon la variable
        stats_match = {
            'Home_Team': club_domicile,
            'Away_Team': club_exterieur,
            'Date': date_m,
            'Temp_Moy_C': round(df_match['temperature_2m'].mean(), 1),
            'Humidite_Moy_%': round(df_match['relative_humidity_2m'].mean(), 1),
            'Pluie_Tot_mm': round(df_match['precipitation'].sum(), 1),
            'Neige_Tot_cm': round(df_match['snowfall'].sum(), 1),
            'Vent_Moy_kmh': round(df_match['wind_speed_10m'].mean(), 1),
            'Rafale_Max_kmh': round(df_match['wind_gusts_10m'].max(), 1)
        }
        
        return stats_match
    else:
        print(f"[Météo] Erreur API le {date_match}: {response.text}")
        return {}

# ==========================================
# 3. EXÉCUTION SUR TON DATAFRAME
# ==========================================

df_matchs = pd.read_csv("data/fbref_games_2024.csv")
df_matchs = pd.concat([df_matchs, pd.read_csv("data/fbref_games2025.csv")], ignore_index=True)
df_matchs = df_matchs[df_matchs['date'] <= "2026-04-28"]

venues = pd.read_csv("data/pl_events.csv")[['venue_name', 'venue_city']].drop_duplicates().set_index('venue_name')['venue_city'].to_dict()
venues['Goodison Park'] = 'Liverpool'  # Correction manuelle pour Everton
venues['The City Ground'] = "West Bridgford"  # Correction manuelle pour Nottingham Forest


historique_meteo = []

for index, row in df_matchs.iterrows():
    venue = row['venue']
    club_domicile = row['home_team']
    club_exterieur = row['away_team']
    date_m = row['date']
    heure_m = row['time']
    
    # Sécurité : vérifier que l'heure n'est pas manquante (parfois le cas sur FBRef)
    if pd.isna(heure_m):
        historique_meteo.append({})
        continue
        
    # Nettoyage de l'heure si elle contient des lettres (ex: "15:00 (Local)")
    heure_propre = str(heure_m).split(' ')[0] 
    
    try : 
        city = venues[venue]
    except KeyError:
        city = venue
    
    # 1. Coordonnées
    lat, lon = obtenir_coordonnees(venue, city)
    
    # 2. Météo complète
    meteo_dict = obtenir_meteo_match_complete(club_domicile, club_exterieur, date_m, lat, lon, date_m, heure_propre)
    
    # 3. Ajout à la liste
    historique_meteo.append(meteo_dict)

# Fusion
df_meteo_ajoutee = pd.DataFrame(historique_meteo)


In [64]:
df_meteo_ajoutee.to_csv("data/historique_meteo.csv", index=False)